# OCR reward

Flow-GRPO uses an OCR-based reward for the text rendering task:

$$r = \max\left(1 - \frac{N_e}{N_{ref}},\ 0\right)$$

$N_e$ is the edit distance between the OCR output and the target, $N_{ref}$ is the target length.

Before training against it I want to see how it scores different kinds of mistakes. No model and no OCR here, I just type the OCR outputs by hand.

## Edit distance

Plain Levenshtein.

In [1]:
def edit_distance(a: str, b: str) -> int:
    previous = list(range(len(b) + 1))
    for i, char_a in enumerate(a, start=1):
        current = [i]
        for j, char_b in enumerate(b, start=1):
            current.append(
                min(
                    previous[j] + 1,
                    current[j - 1] + 1,
                    previous[j - 1] + (char_a != char_b),
                )
            )
        previous = current
    return previous[-1]


for a, b in [("kitten", "sitting"), ("bread", "bresd"), ("bread", ""), ("bread", "bread")]:
    print(f"{a!r:>10} -> {b!r:<10} {edit_distance(a, b)}")

  'kitten' -> 'sitting'  3
   'bread' -> 'bresd'    1
   'bread' -> ''         5
   'bread' -> 'bread'    0


## Reward

This is what the official code does ([flow_grpo/ocr.py](https://github.com/yifan123/flow_grpo/blob/main/flow_grpo/ocr.py)):

- target is the text inside the first pair of quotes in the prompt
- both strings are lowercased and spaces are removed
- if the target is a substring of the OCR text, distance is 0
- otherwise it is the edit distance between the whole OCR text and the target, capped at the target length

In [2]:
def squash(text: str) -> str:
    return text.replace(" ", "").lower()


def official_reward(prompt: str, ocr_text: str) -> float:
    target = squash(prompt.split('"')[1])
    read = squash(ocr_text)
    if target in read:
        distance = 0
    else:
        distance = min(edit_distance(read, target), len(target))
    return 1 - distance / len(target)

## Some cases

In [3]:
PROMPT = 'A bakery window with "Fresh Bread" painted on the glass'

CASES = [
    ("Fresh Bread", "nothing"),
    ("Fresh Bresd", "one wrong letter"),
    ("FRESH BREAD", "wrong case"),
    ("FreshBread", "missing space"),
    ("Fresh Bread Bakery since 1987", "extra text in the scene"),
    ("Fresh Bread Fresh Bresd", "text drawn twice, second copy wrong"),
    ("Fresh", "second word missing"),
    ("Fresh Bresd Bakery since 1987", "one wrong letter and extra text"),
    ("", "no text at all"),
]

print(f"{'OCR output':<33}{'what is wrong':<38}{'reward':>6}")
for ocr_text, problem in CASES:
    print(f"{ocr_text!r:<33}{problem:<38}{official_reward(PROMPT, ocr_text):>6.2f}")

OCR output                       what is wrong                         reward
'Fresh Bread'                    nothing                                 1.00
'Fresh Bresd'                    one wrong letter                        0.90
'FRESH BREAD'                    wrong case                              1.00
'FreshBread'                     missing space                           1.00
'Fresh Bread Bakery since 1987'  extra text in the scene                 1.00
'Fresh Bread Fresh Bresd'        text drawn twice, second copy wrong     1.00
'Fresh'                          second word missing                     0.50
'Fresh Bresd Bakery since 1987'  one wrong letter and extra text         0.00
''                               no text at all                          0.00


Wrong case, missing space, extra text and a duplicated copy all get 1.0. Case and spaces are dropped before comparing, and because of the substring check one clean copy anywhere in the image is enough.

The row I did not expect is "one wrong letter and extra text". One typo alone costs 0.1. One typo plus other text in the image costs everything, because without a clean copy the fallback compares the target against the full OCR string.

## How fast it falls off

Same two renderings, with more and more unrelated text around them.

In [4]:
FILLER = "bakerysince1987"

print(f"{'extra characters':<18}{'clean copy':>12}{'one wrong letter':>18}")
for extra in (0, 1, 2, 4, 6, 9, 12, 15):
    clean = official_reward(PROMPT, "Fresh Bread" + FILLER[:extra])
    typo = official_reward(PROMPT, "Fresh Bresd" + FILLER[:extra])
    print(f"{extra:<18}{clean:>12.2f}{typo:>18.2f}")

extra characters    clean copy  one wrong letter
0                         1.00              0.90
1                         1.00              0.80
2                         1.00              0.70
4                         1.00              0.50
6                         1.00              0.30
9                         1.00              0.00
12                        1.00              0.00
15                        1.00              0.00


With a clean copy extra text is free. With a typo each extra character costs the same as another typo, so 9 extra characters put a 10-character target at zero.

Prompts where the scene has its own text (storefront, newspaper) will give noisy rewards. A model could also learn to get around this by drawing the target twice or by dropping all other text. Need to check for that after training.

## Target length

In [5]:
print(f"{'target length':<15}{'reward after one wrong letter':>30}")
for length in (4, 6, 10, 20, 30):
    target = "a" * length
    prompt = f'A sign that says "{target}"'
    print(f"{length:<15}{official_reward(prompt, 'b' + target[1:]):>30.3f}")

target length   reward after one wrong letter
4                                       0.750
6                                       0.833
10                                      0.900
20                                      0.950
30                                      0.967


30 characters with a typo still gets 0.97, and the text is still wrong. Mean reward does not tell how often the text is fully correct, and it shifts with the mix of target lengths.

## Without the cliff

Match the target against the best substring of the OCR text instead of the whole string.

In [6]:
def substring_distance(target: str, text: str) -> int:
    # first row is zeros so the match can start anywhere in text,
    # min of the last row so it can end anywhere
    previous = [0] * (len(text) + 1)
    for i, char_t in enumerate(target, start=1):
        current = [i]
        for j, char_x in enumerate(text, start=1):
            current.append(
                min(
                    previous[j] + 1,
                    current[j - 1] + 1,
                    previous[j - 1] + (char_t != char_x),
                )
            )
        previous = current
    return min(previous)


def aligned_reward(prompt: str, ocr_text: str) -> float:
    target = squash(prompt.split('"')[1])
    return 1 - substring_distance(target, squash(ocr_text)) / len(target)


print(f"{'OCR output':<33}{'official':>10}{'aligned':>9}")
for ocr_text, _ in CASES:
    official = official_reward(PROMPT, ocr_text)
    aligned = aligned_reward(PROMPT, ocr_text)
    print(f"{ocr_text!r:<33}{official:>10.2f}{aligned:>9.2f}")

OCR output                         official  aligned
'Fresh Bread'                          1.00     1.00
'Fresh Bresd'                          0.90     0.90
'FRESH BREAD'                          1.00     1.00
'FreshBread'                           1.00     1.00
'Fresh Bread Bakery since 1987'        1.00     1.00
'Fresh Bread Fresh Bresd'              1.00     1.00
'Fresh'                                0.50     0.50
'Fresh Bresd Bakery since 1987'        0.00     0.90
''                                     0.00     0.00


Same everywhere except the typo plus extra text case: 0.9 instead of 0. It still ignores case, spaces and extra text.

## Notes for later

- keep the official version to compare with the paper
- try the substring version as the training reward
- need strict metrics that are never trained on: exact match with case, CER over the full OCR string
- report results by target length
- after training look for duplicated text and case changes